the current state - 
derivatives of cat12 processing for: T1w scans, (optional: manually_labeled == y )
metadata (row = scan, all protocols)

What this script should do: 
create: long_format:
1) for all cat12 derivatives (marked with subject_id and session_id), extract volumes 
2) merge with metadata df to have all relevant information in long format

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matlab.engine
import re

In [ ]:
def read_volumes(subject_id : str, session_id : str, atlas_name : str, bids_derivatives_dir : str) -> pd.DataFrame:
    """
    Reads volumes of gray matter, white matter, and CSF from a CAT12 preprocessing
    .mat file for a single subject and session.
    Work for a single scan (subject + session).

    Parameters
    ----------
    subject_id : str
        The subject ID (e.g., 'sub-ls012').
    session_id : str
        The session ID (e.g., 'ses-20070328').
    atlas_name : str
        The name of the atlas (e.g., 'schaefer2018tian2020_400_7').
    bids_derivatives_dir : str
        The path to the CAT12 derivatives folder.

    Returns
    -------
    results_df : pd.DataFrame
        A DataFrame containing the extracted regional volume data, for all regions in the specified atlas. Contains gray matter, white matter, and CSF volumes in cubic millimeters (mm³).
        Returns an empty DataFrame if the file or atlas data is not found.
    """
    # Initialize the MATLAB engine
    print("Starting MATLAB engine...")
    eng = matlab.engine.start_matlab()
    print("MATLAB engine started.")

    # Initialize results_df to an empty DataFrame, ensuring something is always returned
    results_df = pd.DataFrame()
    
    
    try:
        results = []
        
        # Add 'sub-' and 'ses-' prefixes if they are not already present
        if not subject_id.startswith('sub-'):
            subject_id = f'sub-{subject_id}'
        if not session_id.startswith('ses-'):
            session_id = f'ses-{session_id}'

        subject_path = os.path.join(bids_derivatives_dir, subject_id)
        session_path = os.path.join(subject_path, session_id, 'anat')
        
        # The filename for the CAT12 ROI file
        mat_file_name = f'catROI_{subject_id}_{session_id}_T1w.mat'
        mat_file_path = os.path.join(session_path, mat_file_name)

        if not os.path.exists(mat_file_path):
            print(f"Error: Missing MAT file at {mat_file_path}.")
            # The function will return the empty results_df here
        else:
            print(f"Found MAT file at {mat_file_path}. Processing...")
            # Load the .mat file into a MATLAB structure
            s = eng.load(str(mat_file_path))

            # Access the main CAT12 data structure
            all_data = s.get("S")
            
            # A list of keys for the volumes we want to extract
            KEYS = ["Vgm", "Vwm", "Vcsf"]

            # Access the specific atlas data and the volume data within it
            # Using .get() with a default value prevents errors if the atlas is missing
            atlas_data = all_data.get(atlas_name)
            
            if atlas_data and atlas_data.get("data"):
                atlas_volume_data = atlas_data.get("data")
                
                vgm, vwm, vcsf = [atlas_volume_data.get(key) for key in KEYS]
                
                # Get the region IDs and format them with the atlas name
                region_ids = np.array(atlas_data.get("ids")).flatten()
                region_ids_formatted = [f"{atlas_name}_{int(roi)}" for roi in region_ids]
                
                # Flatten the volume arrays and convert to cubic millimeters (mm^3)
                gm_vols = np.array(vgm).flatten() * 1000
                wm_vols = np.array(vwm).flatten() * 1000
                csf_vols = np.array(vcsf).flatten() * 1000

                # Append the extracted data to the results list
                for i in range(len(region_ids_formatted)):
                    results.append({
                        'subject_id': subject_id,
                        'session_id': session_id,
                        'atlas_name': atlas_name,
                        'region_label': region_ids_formatted[i],
                        'gm_volume_mm3': gm_vols[i],
                        'wm_volume_mm3': wm_vols[i],
                        'csf_volume_mm3': csf_vols[i]
                    })
                
                # Create the DataFrame
                results_df = pd.DataFrame(results)
                print(f"Successfully extracted volumes for {len(results_df)} rows of data.")
            else:
                print(f"Warning: Atlas '{atlas_name}' or its 'data' key not found in the MAT file.")
                
    finally:
        # It's important to stop the MATLAB engine when you are done
        if 'eng' in locals():
            eng.quit()
            print("\nMATLAB engine stopped.")

    return results_df


def extract_tiv_from_single_xml(subj_id : str, scan_date : str) -> float:
    """
    Extracts TIV from a single XML file.
    
    Parameters:
    - subj_id: Subject ID (e.g., 'ls012')
    - scan_date: Session date (e.g., '20070328')
    
    Returns:
    - The TIV value as a float, or None if the value cannot be extracted.
    """
    
    # Construct the XML file path
    xml_file_path = f"/home/gaia/Projects/HardDriveOutput/BIDS_thirties/derivatives/CAT12.9_2577/sub-{subj_id}/ses-{scan_date}/anat/cat_sub-{subj_id}_ses-{scan_date}_T1w.xml"

    # ====================================================================================
    # SCRIPT LOGIC (MODIFIED FOR SINGLE TIV EXTRACTION)
    # ====================================================================================

    # Extract subject and session from the filename for later use
    xml_pattern = re.compile(r'cat_sub-(?P<subject>\w+)_ses-(?P<session>\w+)_.*\.xml')
    match = xml_pattern.search(xml_file_path)

    if not match:
        print("Error: Could not extract subject and session from the XML file path.")
        return None

    # Define paths for MATLAB script and output file
    template = "/home/gaia/Projects/legacy_data/my_master/cat12_tiv_template.m"
    filled_template = "/home/gaia/Projects/legacy_data/my_master/cat12_tiv.m"
    out_file = "/home/gaia/Projects/legacy_data/my_master/TIV.txt"
    
    try:
        with open(template, "r") as f:
            template_content = f.read()

        template_content = template_content.replace("$XMLS", f"'{xml_file_path}'").replace("$OUT_FILE", out_file)

        with open(filled_template, "w") as f:
            f.write(template_content)

        SPM_PATH = '/media/storage/master/matlab_R2025a_Prerelease_Linux/spm12'
        CAT12_PATH = "/media/storage/master/matlab_R2025a_Prerelease_Linux/spm12/toolbox/cat12"
        
        # Build and execute the MATLAB command
        cmd = " ".join(
                    [
                        "/usr/local/MATLAB/R2025a/bin/matlab",
                        "-nodisplay",
                        "-nosplash",
                        "-nodesktop",
                        "-r",
                        '"',
                        f"addpath {SPM_PATH} {CAT12_PATH};",
                        f"try, run('{filled_template}'); catch; end; exit;",
                        '"',
                    ]
                )
        os.system(cmd)

        # Read the TIV from the output file
        tiv_data = pd.read_csv(out_file, header=None).values.flatten()
        
        # Return the first (and only) TIV value
        tiv_value = tiv_data[0]
        print(f"TIV value extracted: {tiv_value}")
        return tiv_value
        
    except FileNotFoundError as e:
        print(f"Error: A required file was not found. Details: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

: 

In [ ]:
# go over the derivatives and create a long format dataframe
import pandas as pd

# Assuming read_volumes and extract_tiv_from_single_xml are defined elsewhere
# from your_module import read_volumes, extract_tiv_from_single_xml 

if __name__ == "__main__":
    # --- Step 1: Define file paths and parameters for the batch process ---
    BIDS_DERIVATIVES_DIR = '/home/gaia/Projects/HardDriveOutput/BIDS_over_thirties/derivatives/CAT12.9_2577'
    OUTPUT_CSV_PATH = '/home/gaia/Projects/legacy_data/refactored_project/data/processed/HardDrive_over_thirties_long_format.csv'
    ATLAS_NAME = 'schaefer2018tian2020_400_7'

    # Read the CSV file and explicitly set the data types for specific columns
    metadata = pd.read_csv(
        '/home/gaia/Projects/legacy_data/my_master/HardDrive_over_thirties_full_metadata_filtered.csv',
        dtype={'unique_id': str, 'session_id_date': str})
    
    metadata['unique_id'] = metadata['unique_id'].astype(str)
    metadata['session_id_date'] = metadata['session_id_date'].astype(str)

    # List of ALL metadata columns to be added to the long format.
    # We exclude the columns already present in vol_df (like subject/session IDs) 
    # but include all others that define the subject/scan.
    # Based on your provided column index, let's select the relevant ones:
    METADATA_COLS_TO_ADD = [
        'sex', 'dob', 'scan_date', 'scan_time', 'age_at_scan', 'weight',       
        'protocol', 'institute', 'manufacturer', 'file_path', 
        'age_in_years', 'estimated_critical_info', 'birth_year'
    ]


    # DataFrames to hold regional volumes from each subject
    long_format = pd.DataFrame()
    
    # --- Step 2: Loop through each subject/session and process the data ---
    print(f"Starting batch processing for {len(metadata)} subjects...")
    print("-" * 50)
    
    for index, row in metadata.iterrows(): 
        subj_id = row['unique_id']
        session_id = row['session_id_date']
        
        # Call the function to get the data for one subject
        vol_df = read_volumes(subj_id, session_id, ATLAS_NAME, BIDS_DERIVATIVES_DIR)

        # Process the data if the DataFrame is not empty
        if not vol_df.empty:
            
            # 1. Filter the current subject's metadata using ALL the desired columns
            # We use row[METADATA_COLS_TO_ADD].to_dict() to extract all the columns.
            subject_meta = row[METADATA_COLS_TO_ADD].to_dict()
            
            # 2. Use pandas melt to transform the DataFrame into long format
            subject_long_df = pd.melt(
                vol_df,
                id_vars=['subject_id', 'session_id', 'atlas_name', 'region_label'],
                value_vars=['gm_volume_mm3', 'wm_volume_mm3', 'csf_volume_mm3'],
                var_name='tissue_type',
                value_name='volume_mm3'
            )
            
            # 3. Add ALL extracted metadata columns to the long-format DataFrame
            # Loop through the dictionary and add each column, which is cleaner than
            # writing assignment lines for every column.
            for col_name, col_value in subject_meta.items():
                subject_long_df[col_name] = col_value

            # 4. Extract TIV and add it to the subject's long-format DataFrame
            subject_long_df['tiv'] = extract_tiv_from_single_xml(subj_id, session_id)
            
            # 5. Now, concatenate the long-format data to the main DataFrame
            long_format = pd.concat([long_format, subject_long_df], ignore_index=True)

            print(f"Processed data for **{subj_id} / {session_id}** successfully.")
        
    # --- Step 3: Save the final output after the loop is complete ---
    if not long_format.empty:
        # Reorder columns for better readability (optional)
        # Construct the desired column order dynamically
        ID_COLS = ['subject_id', 'session_id']
        VOLUME_COLS = ['atlas_name', 'region_label', 'tissue_type', 'volume_mm3', 'tiv']
        
        # The final column order: IDs + All Metadata + Volume/Region info
        desired_cols = ID_COLS + METADATA_COLS_TO_ADD + VOLUME_COLS
        
        # Ensure we only try to select columns that exist in the DataFrame
        final_cols = [col for col in desired_cols if col in long_format.columns]

        long_format = long_format[final_cols]
        
        long_format.to_csv(OUTPUT_CSV_PATH, index=False)
        print("\n" + "=" * 50)
        print("✅ **Successfully created and saved the master DataFrame in long format.**")
        print("=" * 50)
        print(f"Final DataFrame shape: {long_format.shape}")
        print("\n**First 5 rows with all metadata columns:**")
        print(long_format.head())
    else:
        print("\nNo data was processed. Please check your inputs.")

: 

In [ ]:
# rows without important info - sex, age_in_years, dob, birth_year
print(f"rows without sex: {metadata['sex'].isna().sum()}")
print(f"rows without age_in_years: {long_format['age_in_years'].isna().sum()}")
print(f"rows without dob: {long_format['dob'].isna().sum()}")
print(f"rows without birth_year: {long_format['birth_year'].isna().sum()}")

: 

In [ ]:
rows_without_sex = metadata[metadata['sex'].isna()]

: 